# Data Enhancing – Metallicity (Materials Project)

Raw database → one labeled row per unique composition → 142 composition descriptors (Magpie + stoichiometry + valence orbitals).

* **Source:** Materials Project (Matbench `matbench_mp_is_metal`)
* **Target:** is_metal (1 = metal, 0 = non-metal)

In [1]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
TASK = "metallicity"
RAW = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"

In [2]:
# Cell 1: Load the raw database
raw = pd.read_parquet(RAW / "matbench/matbench_mp_is_metal.parquet")
print(f"raw entries: {len(raw):,}")
raw.head()

raw entries: 106,113


,formula,nsites,energy_proxy_volume_pa,is_metal
0,KMnO2,16,13.836878,False
1,Cr3Ni,4,11.367855,True
2,CsRbAs,3,52.035996,True
3,SiSn,2,27.946089,False
4,Ca3P2,5,29.944258,False


In [3]:
# Cell 2: Simple stats
raw.describe(include='all').T.head(15)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
formula,106113,78164,Li9Mn2Co5O16,702,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nsites,106113.0,NaN,NaN,NaN,30.015022,29.940163,1.0,10.0,20.0,40.0,444.0
energy_proxy_volume_pa,106113.0,NaN,NaN,NaN,17.76492,8.840762,5.277238,12.182773,15.025509,21.430871,741.7335
is_metal,106113,2,False,59962,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Cell 3: One label per unique composition (reduced formula, polymorph rule)
import prepare_datasets as P
lab = P.labeled_metallicity()
print(f"unique compositions: {len(lab):,}")
print(lab["target"].value_counts().rename({0: "negative", 1: "positive"}))
lab.head()

unique compositions: 77,432
target
negative    40530
positive    36902
Name: count, dtype: int64


,formula,target,n_polymorphs
0,Ac,1,4
1,Ac2AgIr,1,1
2,Ac2AgPb,1,1
3,Ac2CdGa,1,1
4,Ac2CdGe,1,1


In [5]:
# Cell 4: Composition featurisation (vectorised Magpie, identical to matminer)
from featurize import featurize_formulas, feature_names
examples = pd.Series(["LiFePO4", "Fe2O3", "GaAs", "NdFeB", "SrTiO3"])
X, ok = featurize_formulas(examples)
X.index = examples
print(len(feature_names()), "descriptors")
X[["MagpieData mean Electronegativity", "MagpieData range Electronegativity", "MagpieData mean CovalentRadius", "MagpieData mean NValence", "MagpieData mean GSmagmom", "0-norm", "frac d valence electrons"]]

142 descriptors


,MagpieData mean Electronegativity,MagpieData range Electronegativity,MagpieData mean CovalentRadius,MagpieData mean NValence,MagpieData mean GSmagmom,0-norm,frac d valence electrons
LiFePO4,2.680,2.46,90.142857,5.428571,0.301523,4.0,0.157895
Fe2O3,2.796,1.61,92.400000,6.800000,0.844265,2.0,0.352941
GaAs,1.995,0.37,120.500000,14.000000,0.000000,2.0,0.714286
NdFeB,1.670,0.90,139.000000,5.666667,0.703554,3.0,0.352941
SrTiO3,2.562,2.49,110.600000,4.800000,0.000005,3.0,0.083333


In [6]:
# Cell 5: Check against matminer's reference implementation
from matminer.featurizers.composition import ElementProperty, Stoichiometry, ValenceOrbital
from pymatgen.core import Composition
ep, st, vo = ElementProperty.from_preset("magpie"), Stoichiometry(), ValenceOrbital(props=["frac"])
ref = np.array([ep.featurize(Composition(f)) + st.featurize(Composition(f)) + vo.featurize(Composition(f)) for f in examples], float)
print("max |ours - matminer| =", np.nanmax(np.abs(X.values - ref)))

max |ours - matminer| = 2.2737367544323206e-13


In [7]:
# Cell 6: Featurise the whole labeled set and the GNoME unlabeled pool
import time
t = time.time()
feats, ok = featurize_formulas(lab['formula'])
print(f"featurised {ok.sum():,} compositions in {time.time()-t:.1f}s")
df = pd.read_parquet(PROC / "metallicity_features.parquet")
gn = pd.read_parquet(PROC / "unlabeled_gnome_features.parquet", columns=["formula"])
print(f"saved labeled table: {df.shape} | GNoME unlabeled pool: {len(gn):,}")

featurised 77,432 compositions in 5.3s


saved labeled table: (77432, 145) | GNoME unlabeled pool: 554,054
